In [ ]:
import os
import re
import fitz  # PyMuPDF

/home/yan/Documents/GitHub/.venv/lib/python3.13/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [ ]:
def clean_text_block(text):

    #Remove quebras de linha no meio de frases (comuns em extração de PDF), mas preserva tópicos/bullets e parágrafos.
    if not text:
        return ""
    # Transforma bullets do PDF (•) em listas Markdown (-)
    text = re.sub(r'^[•\u2022]\s*', '- ', text, flags=re.MULTILINE)
    # Substitui quebra de linha única por espaço em parágrafos normais
    text = re.sub(r'(?<!\n)\n(?!\n|- )', ' ', text)
    return text.strip()

def extract_clean_text(doc):

    #Extrai o texto preservando a separação visual dos blocos do PDF
    full_blocks = []
    for page in doc:
        blocks = page.get_text("blocks")
        for b in blocks:
            full_blocks.append(b[4].strip())
    return "\n\n".join(full_blocks)

def extract_metadata_and_sections(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        full_text = extract_clean_text(doc)
    except Exception as e:
        print(f"Erro ao ler {pdf_path}: {e}")
        return None

    filename = os.path.basename(pdf_path).replace('.pdf', '')

    # Pega o Nome do Pokémon (Descarta prefixos como '1_', '10_', 'regmb_')
    name_match = re.search(r'(?:reg[a-z0-9]+_|\d+_)*([A-Z][a-zA-Z]*(?:_[A-Za-z0-9]+)*)', filename)
    if name_match:
        pokemon_name = name_match.group(1).replace('_', ' ').strip()
    else:
        pokemon_name = "Pokémon Desconhecido"

    # Pega o Formatos (Limpa para não pegar o "Written by...")
    formats_match = re.search(r'Formats:\s*(.*?)(?=\nWritten by|\nMega|\nSetup|\nMove|\nNature|\Z)', full_text, re.DOTALL | re.IGNORECASE)
    formats = clean_text_block(formats_match.group(1)) if formats_match else "Não especificado"

    # Pega a Nature (Captura flexível (aceita 'Nature: Adamant', 'Nature\nAdamant', etc.))
    nature_match = re.search(r'Nature:?\s*\n?\s*([A-Z][a-z]+)', full_text, re.IGNORECASE)
    nature = nature_match.group(1) if nature_match else "Não especificada"

    # Level
    level_match = re.search(r'Level:?\s*\n?\s*(\d+)', full_text, re.IGNORECASE)
    level = level_match.group(1) if level_match else "50"

    # Pega a Função Competitiva (Role / Build Name)
    role_match = re.search(r'Written by [^\n]+\n+([^\n]+?)(?=\s*Export|\s*Move 1|\s*Item:|\n)', full_text, re.IGNORECASE)
    if role_match:
        role = clean_text_block(role_match.group(1))
    else:
        fallback_roles = r'(Bulky Attacker|Setup Sweeper|Wallbreaker|Pivot|Cleric|Support|Tank|Sweeper|Lead|Fast Attacker|Offensive Tank)'
        role_fallback = re.search(fallback_roles, full_text, re.IGNORECASE)
        role = role_fallback.group(1) if role_fallback else "Não especificada"

    # Extração das Seções de Texto
    strat_match = re.search(r'Export\n(.*?)(?=\nOther Options|\nTeambuilding|\nChecks and Weaknesses)', full_text, re.DOTALL | re.IGNORECASE)
    main_strat = clean_text_block(strat_match.group(1)) if strat_match else "Estratégia não encontrada."

    oo_match = re.search(r'Other Options\n(.*?)(?=\nTeambuilding|\nChecks and Weaknesses)', full_text, re.DOTALL | re.IGNORECASE)
    other_options = clean_text_block(oo_match.group(1)) if oo_match else "Nenhuma opção alternativa listada."

    # Captura tudo do Teambuilding
    tb_match = re.search(r'(?:Teambuilding|Checks and Weaknesses\.?)\n?(.*)', full_text, re.DOTALL | re.IGNORECASE)
    
    if tb_match:
        raw_tb = tb_match.group(1)
        # Limpa o rodapé (Credits e a tabela de Moves no final do PDF se existirem)
        clean_tb = re.split(r'\n+(?:Credits|Moves)\b', raw_tb, flags=re.IGNORECASE)[0]
        teambuilding = clean_text_block(clean_tb)
    else:
        teambuilding = "Informação de Teambuilding não encontrada."

    # Template Final Markdown
    md_content = f"""# {pokemon_name.title()} — Competitive Strategies

## Identification and formats
- Pokémon: 
{pokemon_name.title()}
- Formats: 
{formats}
- Competitive Role: 
{role.title()}
- Nature: 
{nature.title()}
- Level: 
{level}

## Core strategy and moves
{main_strat}

## Alternative Movements
{other_options}

## Team Building
{teambuilding}
"""
    return md_content

def process_pipeline(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith('.pdf'):
            pdf_path = os.path.join(input_folder, filename)
            
            md_content = extract_metadata_and_sections(pdf_path)
            
            if md_content:
                md_filename = f"{filename.replace('.pdf', '')}.md"
                md_filepath = os.path.join(output_folder, md_filename)
                
                with open(md_filepath, 'w', encoding='utf-8') as f:
                    f.write(md_content)
                print(f"Sucesso: {md_filename} processado!")

if __name__ == "__main__":
    PASTA_ENTRADA = "/home/yan/Documents/GitHub/competitive-poke-ai123/Metagame" 
    PASTA_SAIDA = "/home/yan/Documents/GitHub/competitive-poke-ai123/Metagame/md2"
    
    print("Iniciando extração e reestruturação dos PDFs...")
    process_pipeline(PASTA_ENTRADA, PASTA_SAIDA)
    print("Concluído!")

Iniciando extração e reestruturação dos PDFs...
Sucesso: 48_regma_Gengar _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 16_regma_Pelipper _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 57_regma_Ceruledge _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 24_regmb_Torkoal _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 30_regmb_Garchomp _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 40_regmb_Incineroar _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 44_regmb_Staraptor _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 46_regma_Excadrill _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 39_regmb_Resource - VGC Regulation M-B Viability Rankings _ Smogon Forums.md processado!
Sucesso: 88_regmb_Delphox _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 69_regmb_Mamoswine _ Champions _ Smogon Strategy Pokedex.md processado!
Sucesso: 77_regmb_VGC 2026 Regulation M-B _ Champion